**Projenin Amacı**

Bu projenin temel amacı, yaklaşık 8 milyon satır içeren devasa bir Amazon kullanıcı etkileşim verisini işleyerek, kullanıcıların geçmiş satın alma alışkanlıklarına dayalı bir öneri motoru kurmaktır.

**Spesifik hedefler şunlardır:**

İşbirlikçi Filtreleme (Collaborative Filtering): Ürünlerin içeriğine bakmaksızın, "Bu ürünü alanlar şu ürünleri de beğendi" mantığını kurmak.

Kalite Filtrelemesi: Önerilen ürünler arasında sadece beraber alınanları değil, bu ürünler arasından müşteri memnuniyeti (ortalama puan) en yüksek olanları öne çıkarmak.


<img src='https://m.media-amazon.com/images/I/61zYzaUw31L.png' width=600>

In [1]:
# Verileri tablolar halinde düzenlemek, analiz etmek ve filtrelemek için kullanılır.
import pandas as pd

# Metinlerdeki kelimeleri, önem derecelerine göre matematiksel sayılara (vektörlere) dönüştürür.
from sklearn.feature_extraction.text import TfidfVectorizer

# Sayısal hale gelmiş metinler arasındaki benzerliği, aralarındaki açıyı ölçerek hesaplar.
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
#veriyi yüklüyoruz

df=pd.read_csv('ratings_Electronics.csv')

# EDA

In [3]:
#ilk 5 satır göster
df.head()

,AKM1MP6P0OYPR,0132793040,5.0,1365811200
0,A2CX7LUOHB2NDG,0321732944,5.0,1341100800
1,A2NWSAGRHCP8N5,0439886341,1.0,1367193600
2,A2WNBOD3WNDNKT,0439886341,3.0,1374451200
3,A1GI0U4ZRJA8WN,0439886341,1.0,1334707200
4,A1QGNMC6O1VW39,0511189877,5.0,1397433600


In [4]:
# Sütun isimlerini biz belirliyoruz
#Kimin aldığı (user_id),
#Neyi aldığı (product_id),
#Kaç puan verdiği (rating)
#Hangi gün ve saatte puan verdiği (timestamp)
sutunlar = ['user', 'item', 'rating', 'timestamp']

In [5]:
# header=None ile ilk satırın başlık olmadığını belirtiyoruz
df = pd.read_csv('ratings_Electronics.csv', names=sutunlar, header=None)

In [6]:
#ilk 5 satır göster
df.head()

,user,item,rating,timestamp
0,AKM1MP6P0OYPR,0132793040,5.0,1365811200
1,A2CX7LUOHB2NDG,0321732944,5.0,1341100800
2,A2NWSAGRHCP8N5,0439886341,1.0,1367193600
3,A2WNBOD3WNDNKT,0439886341,3.0,1374451200
4,A1GI0U4ZRJA8WN,0439886341,1.0,1334707200


In [7]:
#veriler hakkında bilgi
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7824482 entries, 0 to 7824481
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   user       object 
 1   item       object 
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(1), object(2)
memory usage: 238.8+ MB


In [8]:
#kaç satır kaç sütun
df.shape

(7824482, 4)

In [9]:
#boş veri kontrolü
df.isnull().sum()

user         0
item         0
rating       0
timestamp    0
dtype: int64

In [10]:
#timestamp gereksiz sütun olduğu için kaldırıyoruz
df.drop('timestamp', axis=1, inplace=True)

In [11]:
# Her ürünün ortalama puanını hesapla (Öneri sırasında kullanacağız)
avg_ratings = df.groupby('item')['rating'].mean().reset_index()
avg_ratings.columns = ['item', 'average_rating']

In [12]:
# Bu hesaplanan puanları ana df tablosuna 'item' üzerinden birleştir
df = pd.merge(df, avg_ratings, on='item')

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7824482 entries, 0 to 7824481
Data columns (total 4 columns):
 #   Column          Dtype  
---  ------          -----  
 0   user            object 
 1   item            object 
 2   rating          float64
 3   average_rating  float64
dtypes: float64(2), object(2)
memory usage: 238.8+ MB


In [14]:
def amazon_puanli_oner(urun_id):
    # 1. Bu ürünü alan kullanıcıları bul
    kullanicilar = df[df['item'] == urun_id]['user'].unique()
    
    # 2. Bu kullanıcıların aldığı diğer ürünleri getir (Ana ürünü listeden çıkar)
    onerilenler_df = df[df['user'].isin(kullanicilar) & (df['item'] != urun_id)]
    
    if onerilenler_df.empty:
        return pd.DataFrame(columns=['item', 'average_rating']) # Boşsa direkt dön
    
    # 3. Önerilen ürünlerin popülerliğini ve puanını hesapla
    # Sadece 'item' ve 'average_rating' sütunlarını al ve en iyileri seç
    sonuc = onerilenler_df[['item', 'average_rating']].drop_duplicates()
    
    # Puanı en yüksek olan ve en çok tercih edilenleri sırala
    sonuc = sonuc.sort_values(by='average_rating', ascending=False)
    
    return sonuc.head(3).reset_index(drop=True)

In [15]:
df['item'].value_counts()

item
B0074BW614    18244
B00DR0PDNE    16454
B007WTAJTO    14172
B0019EHU8G    12285
B006GWO5WK    12226
              ...  
B004WL91KI        1
B004WL9FK4        1
B004WL9Q2Q        1
B004WL9R8O        1
BT008V9J9U        1
Name: count, Length: 476002, dtype: int64

In [16]:
#Kullanımı:
amazon_puanli_oner('B00AYM9U32')

,item,average_rating
0,B003DS5XDA,5.000000
1,B00955FWDQ,5.000000
2,B000WYVBR0,4.582609


In [17]:
#Kullanımı:
amazon_puanli_oner('B007WTAJTO')

,item,average_rating
0,B00DEB5WAA,5.0
1,B008CZ9P7O,5.0
2,B00EA6902O,5.0


**SONUÇ**

Bu projede İşbirlikçi Filtreleme (Collaborative Filtering) dediğimiz bir mantıkla çalışır. Bizim verdiğimiz urun_id (örneğin bir kulaklık olsun) üzerinden şu yolu izler:

1. Adım (Ortak Müşterileri Bulma): "Bu kulaklığı kimler satın aldı?" sorusuna yanıt arar ve o kullanıcıların listesini çıkarır.

2. Adım (Diğer Tercihleri İzleme): "Bu kullanıcılar kulaklık dışında başka neler aldılar?" diyerek o kişilerin tüm alışveriş geçmişine bakar.

3. Adım (Kalite Filtresi): Bu ortak müşterilerin aldığı yüzlerce ürün arasından, en yüksek puana (average_rating) sahip olan ilk 3 tanesini seçer ve temiz bir liste halinde bize sunar.

In [18]:
import joblib

# Sisteminin çalışması için gereken en güncel tabloları bir sözlükte topla
amazon_recommender_package = {
    'main_data': df,            # Kullanıcı-Ürün eşleşmelerinin olduğu ana tablo
    'avg_ratings': avg_ratings  # Ürünlerin ortalama puanlarının olduğu tablo
}

# Tek bir dosya olarak kaydet
joblib.dump(amazon_recommender_package, 'amazon_recommender_package.pkl')

print("Öneri sistemi verileri 'amazon_recommender_package.pkl' adıyla kaydedildi!")

Öneri sistemi verileri 'amazon_recommender_package.pkl' adıyla kaydedildi!
